In [1]:
import pandas as pd
#import matplotlib.pyplot as plt
from collections import defaultdict
import json
import re
import random
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
from transformers import RobertaTokenizer, RobertaModel, AutoTokenizer

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from datasets import load_dataset

import xgboost as xgb
import matplotlib.pyplot as plt
from scipy.stats import shapiro

from matplotlib.lines import Line2D

c:\Users\Michael\Documents\Projects\MSCthesis\.env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

HE_dataset = load_dataset("openai/openai_humaneval", split="test")
MBPP_dataset = load_dataset("mbpp")

task_to_prompt = {ex["task_id"]: ex["prompt"] for ex in HE_dataset}
task_to_prompt.update({"mbpp/" + str(ex["task_id"]): ex["text"] for ex in MBPP_dataset["test"]})
task_to_prompt.update({"mbpp/" + str(ex["task_id"]): ex["text"] for ex in MBPP_dataset["train"]})
task_to_prompt.update({"mbpp/" + str(ex["task_id"]): ex["text"] for ex in MBPP_dataset["validation"]})
task_to_prompt.update({"mbpp/" + str(ex["task_id"]): ex["text"] for ex in MBPP_dataset["prompt"]})


In [47]:
#load
with open("results/aggregated_data_labeled.json", "r") as f:
    loaded_data = json.load(f)
df = pd.DataFrame(loaded_data)

# Plottings

# Bert Embedding

In [52]:
MODEL_NAME = "microsoft/codebert-base"
#MODEL_NAME = "./bert_model"
#MODEL_NAME = "./bert_model_mbpptuned_fix"
MAX_LEN = 512
K_NEIGHBORS = 1
TEST_SIZE = 0.2
RANDOM_STATE = 42

device = "cuda" if torch.cuda.is_available() else "cpu"
#device = "cpu"
print(f"[info] Using device: {device}")



codebert_tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
codebert_model = RobertaModel.from_pretrained(MODEL_NAME).to(device).eval()

def embed_texts(texts, batch_size=100):
    embeds = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding prompts", disable=True):
        batch = texts[i:i+batch_size]
        inputs = codebert_tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
        with torch.no_grad():
            out = codebert_model(**inputs).last_hidden_state
            mask = inputs["attention_mask"].unsqueeze(-1)
            pooled = (out * mask).sum(dim=1) / mask.sum(dim=1)
        embeds.append(pooled.cpu().numpy())
    return np.vstack(embeds)

[info] Using device: cuda


# MBPP HE

In [65]:
#Aggregated train test split for direct model prediction
def select_winner(group):

    if group["correct"].sum() == 1:
        winner = group[group["correct"]].iloc[0]
    else:
        # Lowest energy if both correct/incorrect
        winner = group.loc[group["avg_energy"].idxmin()]

    return winner


agg_df = (
    df.groupby("task_id", group_keys=False)
      .apply(select_winner)
      .loc[:, ["task_id", "prompt", "model"]]
      .rename(columns={"model": "correct_model"})
      .reset_index(drop=True)
)

print(agg_df["correct_model"].value_counts())

correct_model
deepseek-ai/deepseek-coder-1.3b-instruct    727
Qwen/Qwen2.5-Coder-3B-Instruct              411
Name: count, dtype: int64


C:\Users\Michael\AppData\Local\Temp\ipykernel_129340\4247059413.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(select_winner)


# Router Eval

In [90]:
def evaluate_router_humaneval(eval_dataset, candidate_models, router_func, router, mode="router"):
    correct_count = 0
    total_energy = 0.0
    correct_weak = 0
    energy_weak = 0.0
    correct_strong = 0
    energy_strong = 0.0
    
    n = len(eval_dataset)

    for i in range(len(eval_dataset)):
        prompt = eval_dataset.iloc[i]["prompt"]
        task_id = eval_dataset.iloc[i]["task_id"]

        best_model = None
        if mode == "router":
            best_model, _ = router_func(prompt, candidate_models, router)
        elif mode == "oracle":
            best_model = eval_dataset.iloc[i]["correct_model"]
        elif mode == "strong":
            best_model = candidate_models[0]
        elif mode == "random":
            best_model = random.choice(candidate_models)
        elif mode == "weak":
            best_model = candidate_models[1]

        correct = df.loc[
            (df["task_id"] == task_id) & (df["model"] == best_model),
            "correct"
        ].values
        correct = bool(correct[0]) if len(correct) > 0 else False
        correct_count += int(correct)

        energy = df.loc[
            (df["task_id"] == task_id) & (df["model"] == best_model),
            "avg_energy"
        ].values
        energy = float(energy[0]) if len(energy) > 0 else 0.0
        total_energy += energy

        #print(f"Task {task_id} {i}/{n} | Routed to {best_model} | Correct: {correct} | Energy: {energy:.2f}J")
        
        
    for i in range(len(eval_dataset)):
        task_id = eval_dataset.iloc[i]["task_id"]
        best_model = candidate_models[0]

        correct = df.loc[
            (df["task_id"] == task_id) & (df["model"] == best_model),
            "correct"
        ].values
        correct = bool(correct[0]) if len(correct) > 0 else False
        correct_strong += int(correct)

        energy = df.loc[
            (df["task_id"] == task_id) & (df["model"] == best_model),
            "avg_energy"
        ].values
        energy = float(energy[0]) if len(energy) > 0 else 0.0
        energy_strong += energy
        
    for i in range(len(eval_dataset)):
        task_id = eval_dataset.iloc[i]["task_id"]
        best_model = candidate_models[1]

        correct = df.loc[
            (df["task_id"] == task_id) & (df["model"] == best_model),
            "correct"
        ].values
        correct = bool(correct[0]) if len(correct) > 0 else False
        correct_weak += int(correct)

        energy = df.loc[
            (df["task_id"] == task_id) & (df["model"] == best_model),
            "avg_energy"
        ].values
        energy = float(energy[0]) if len(energy) > 0 else 0.0
        energy_weak += energy

        #print(f"Task {task_id} {i}/{n} | Routed to {best_model} | Correct: {correct} | Energy: {energy:.2f}J")
    correct_avg = correct_weak + correct_strong
    energy_avg = energy_weak + energy_strong
    print(f"Baseline: accuracy: {correct_avg/(2*n):.4f} | energy: {energy_avg/(2*n):.2f}J")
    
    #interpolated accuracy and energy
    correct_diff = correct_weak - correct_strong
    energy_diff = energy_weak - energy_strong
    
    factor = (correct_count - correct_weak)/(correct_diff)
    interp_energy = energy_weak + (factor * energy_diff)
    
    
    print(f"Interpolated energy: {interp_energy/n:.2f}J")
    print(f"Strong_model accuracy: {correct_strong/n:.4f} | energy: {energy_strong/n:.2f}J")
    
    return correct_count, total_energy

# K Fold

In [53]:
X = df.drop(columns=["correct"]) 
y = df["correct"]
embeds = embed_texts(X["prompt"].tolist())
groups = df["task_id"]

In [84]:
#global scaler
def route_lr(prompt, category_mapping, router):
    global scaler
    prompt_emb = embed_texts([prompt])
    prompt_emb_scaled = scaler.transform(prompt_emb)#normalize(prompt_emb) 
                
    lr = router
    c_pred = lr.predict(prompt_emb_scaled)[0]
            
    return category_mapping[c_pred], 123


def fold_lr(k=5, random_state=42, fold_accuracies=None, fold_energies=None):
    global scaler
    gkf = GroupKFold(n_splits=k, shuffle=True, random_state=random_state)
    accuracies = []
    energies = []

    for train_idx, test_idx in gkf.split(df, groups=groups):
        df_train, df_test = df.iloc[train_idx], df.iloc[test_idx]



        train_df = (
            df_train.groupby("task_id", group_keys=False)
            .apply(select_winner)
            .loc[:, ["task_id", "prompt", "model"]]
            .rename(columns={"model": "correct_model"})
            .reset_index(drop=True)
        )
        
        test_df = (
            df_test.groupby("task_id", group_keys=False)
            .apply(select_winner)
            .loc[:, ["task_id", "prompt", "model"]]
            .rename(columns={"model": "correct_model"})
            .reset_index(drop=True)
        )
        
        
        mbpp_train_prompts = train_df["prompt"].tolist()
        train_df["correct_model"].tolist()

        mbpp_train_embeddings = embed_texts(mbpp_train_prompts)
        train_df["correct_model_encoded"] = train_df["correct_model"].astype('category').cat.codes
        #print(train_df["correct_model_encoded"].value_counts())

        category_mapping = dict(enumerate(train_df["correct_model"].astype('category').cat.categories))
        
        
        scaler = StandardScaler()
        mbpp_train_embeddings_scaled = scaler.fit_transform(mbpp_train_embeddings)

        lr = LogisticRegression(
            max_iter=2000,
        )


        lr.fit(mbpp_train_embeddings_scaled, train_df["correct_model_encoded"].values)
        
        
        accuracy, avg_energy = evaluate_router_humaneval(test_df, category_mapping, router_func=route_lr, router=lr)
        accuracies.append(accuracy)
        energies.append(avg_energy)
        n = len(test_df)
        print(f"Test set accuracy: {accuracy/n:.4f}, avg energy: {avg_energy/n:.2f}J")
        fold_accuracy = accuracy/n
        fold_energy = avg_energy/n
        fold_accuracies.append(fold_accuracy)
        fold_energies.append(fold_energy)
        
    n = len(df)/2
        
    average_accuracy = sum(accuracies) / n
    average_energy = sum(energies) / n


    print(average_accuracy, average_energy)
    return average_accuracy, average_energy
    

In [61]:
def fold_xgb(e=0.5, k=5, random_state=42, fold_accuracies=None, fold_energies=None):
    gkf = GroupKFold(n_splits=k, shuffle=True, random_state=random_state)
    accuracies = []
    energies = []

    for train_idx, test_idx in gkf.split(df, groups=groups):
        routers = []
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        emb_train = embeds[train_idx]


        # normalize using training split only
        energy_mean = X_train["avg_energy"].mean()
        energy_std  = X_train["avg_energy"].std()
        X_train["energy_norm"] = (X_train["avg_energy"] - energy_mean) / energy_std
        X_test["energy_norm"]  = (X_test["avg_energy"] - energy_mean) / energy_std

        ye_train = y_train - e * X_train["energy_norm"] 
        
        X_train["model"].tolist()
        model_encoded = X_train["model"].astype('category').cat.codes.to_numpy()

        X_train_features = emb_train

        category_mapping = dict(enumerate(X_train["model"].astype('category').cat.categories))
        
        xgb_reg = xgb.XGBRegressor(
            n_estimators=200,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42
        )
        
        xgb_reg2 = xgb.XGBRegressor(
            n_estimators=200,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42
        )
        
        mask_model0 = model_encoded == 0
        mask_model1 = model_encoded == 1

        xgb_reg.fit(X_train_features[mask_model0], ye_train[mask_model0])
        xgb_reg2.fit(X_train_features[mask_model1], ye_train[mask_model1])
        
        routers.append(xgb_reg)
        routers.append(xgb_reg2)
        
        
        X_test_collapsed = X_test.drop_duplicates(subset=["task_id"]).reset_index(drop=True)
        accuracy, avg_energy = evaluate_router_humaneval(X_test_collapsed, category_mapping, router_func=route_xgb, router=routers)
        n = len(X_test_collapsed)
        print(f"Test set accuracy: {accuracy/n:.4f}, avg energy: {avg_energy/n:.2f}J")
        accuracies.append(accuracy)
        energies.append(avg_energy)
        
        fold_accuracy = accuracy/n
        fold_energy = avg_energy/n
        fold_accuracies.append(fold_accuracy)
        fold_energies.append(fold_energy)
        
    n = len(df)/2
        
    average_accuracy = sum(accuracies) / n
    average_energy = sum(energies) / n

    print(average_accuracy, average_energy)
    return average_accuracy, average_energy
    
def route_xgb(prompt, candidate_models, xgb):
    prompt_emb = embed_texts([prompt])

    predictions = []

    for code, model_name in candidate_models.items():
            
        pred = xgb[code].predict(prompt_emb)[0]
        predictions.append((model_name, pred))

    best_model_name, best_score = max(predictions, key=lambda x: x[1])
    return best_model_name, best_score


# Repetition

In [63]:
repetitions = 10

In [70]:
accuracies_lr = []
energies_lr = []
fold_accuracies_lr = []
fold_energies_lr = []

for i in range(repetitions):
    accuracy, energy = fold_lr(k=5, random_state=42+i, fold_accuracies=fold_accuracies_lr, fold_energies=fold_energies_lr)
    
    accuracies_lr.append(accuracy)
    energies_lr.append(energy)

C:\Users\Michael\AppData\Local\Temp\ipykernel_129340\3566723576.py:26: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(select_winner)
C:\Users\Michael\AppData\Local\Temp\ipykernel_129340\3566723576.py:34: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(select_winner)


Baseline: accuracy: 0.5658 | energy: 126.53J
Interpolated energy: 118.24J
Strong_model accuracy: 0.6228 | energy: 162.43J
Test set accuracy: 0.5526, avg energy: 108.29J


C:\Users\Michael\AppData\Local\Temp\ipykernel_129340\3566723576.py:26: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(select_winner)
C:\Users\Michael\AppData\Local\Temp\ipykernel_129340\3566723576.py:34: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(select_winner)


Baseline: accuracy: 0.6360 | energy: 120.33J
Interpolated energy: 100.04J
Strong_model accuracy: 0.7149 | energy: 160.93J
Test set accuracy: 0.5965, avg energy: 100.58J


C:\Users\Michael\AppData\Local\Temp\ipykernel_129340\3566723576.py:26: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(select_winner)
C:\Users\Michael\AppData\Local\Temp\ipykernel_129340\3566723576.py:34: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(select_winner)


...

Baseline: accuracy: 0.6053 | energy: 119.74J
Interpolated energy: 123.13J
Strong_model accuracy: 0.6623 | energy: 163.82J
Test set accuracy: 0.6096, avg energy: 107.76J


C:\Users\Michael\AppData\Local\Temp\ipykernel_129340\3566723576.py:26: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(select_winner)
C:\Users\Michael\AppData\Local\Temp\ipykernel_129340\3566723576.py:34: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(select_winner)


Baseline: accuracy: 0.6278 | energy: 126.07J
Interpolated energy: 121.37J
Strong_model accuracy: 0.7225 | energy: 166.50J
Test set accuracy: 0.6167, avg energy: 117.35J


C:\Users\Michael\AppData\Local\Temp\ipykernel_129340\3566723576.py:26: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(select_winner)
C:\Users\Michael\AppData\Local\Temp\ipykernel_129340\3566723576.py:34: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(select_winner)


Baseline: accuracy: 0.5815 | energy: 126.03J
Interpolated energy: 150.75J
Strong_model accuracy: 0.6035 | energy: 167.23J
Test set accuracy: 0.5947, avg energy: 116.31J
0.5949033391915641 110.08389622144111


In [71]:
accuracies_xgb = []
energies_xgb = []
fold_accuracies_xgb = []
fold_energies_xgb = []

for i in range(repetitions):
    accuracy, energy = fold_xgb(e=0.5, k=5, random_state=42+i, fold_accuracies=fold_accuracies_xgb, fold_energies=fold_energies_xgb)
    
    accuracies_xgb.append(accuracy)
    energies_xgb.append(energy)

C:\Users\Michael\AppData\Local\Temp\ipykernel_129340\3496638334.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train["energy_norm"] = (X_train["avg_energy"] - energy_mean) / energy_std
C:\Users\Michael\AppData\Local\Temp\ipykernel_129340\3496638334.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test["energy_norm"]  = (X_test["avg_energy"] - energy_mean) / energy_std


Baseline: accuracy: 0.5658 | energy: 126.53J
Interpolated energy: 132.05J
Strong_model accuracy: 0.6228 | energy: 162.43J
Test set accuracy: 0.5746, avg energy: 98.66J


C:\Users\Michael\AppData\Local\Temp\ipykernel_129340\3496638334.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train["energy_norm"] = (X_train["avg_energy"] - energy_mean) / energy_std
C:\Users\Michael\AppData\Local\Temp\ipykernel_129340\3496638334.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test["energy_norm"]  = (X_test["avg_energy"] - energy_mean) / energy_std


Baseline: accuracy: 0.6360 | energy: 120.33J
Interpolated energy: 93.27J
Strong_model accuracy: 0.7149 | energy: 160.93J
Test set accuracy: 0.5833, avg energy: 91.93J


C:\Users\Michael\AppData\Local\Temp\ipykernel_129340\3496638334.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train["energy_norm"] = (X_train["avg_energy"] - energy_mean) / energy_std
C:\Users\Michael\AppData\Local\Temp\ipykernel_129340\3496638334.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test["energy_norm"]  = (X_test["avg_energy"] - energy_mean) / energy_std


...

Baseline: accuracy: 0.6053 | energy: 119.74J
Interpolated energy: 116.35J
Strong_model accuracy: 0.6623 | energy: 163.82J
Test set accuracy: 0.6009, avg energy: 94.78J


C:\Users\Michael\AppData\Local\Temp\ipykernel_129340\3496638334.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train["energy_norm"] = (X_train["avg_energy"] - energy_mean) / energy_std
C:\Users\Michael\AppData\Local\Temp\ipykernel_129340\3496638334.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test["energy_norm"]  = (X_test["avg_energy"] - energy_mean) / energy_std


Baseline: accuracy: 0.6278 | energy: 126.07J
Interpolated energy: 110.08J
Strong_model accuracy: 0.7225 | energy: 166.50J
Test set accuracy: 0.5903, avg energy: 91.99J


C:\Users\Michael\AppData\Local\Temp\ipykernel_129340\3496638334.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train["energy_norm"] = (X_train["avg_energy"] - energy_mean) / energy_std
C:\Users\Michael\AppData\Local\Temp\ipykernel_129340\3496638334.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test["energy_norm"]  = (X_test["avg_energy"] - energy_mean) / energy_std


Baseline: accuracy: 0.5815 | energy: 126.03J
Interpolated energy: 76.60J
Strong_model accuracy: 0.6035 | energy: 167.23J
Test set accuracy: 0.5551, avg energy: 107.08J
0.5834797891036907 98.11955695176721


In [72]:
#Save arrays
np.save("Data/accuracies_lr.npy", np.array(accuracies_lr))
np.save("Data/energies_lr.npy", np.array(energies_lr))
np.save("Data/fold_accuracies_lr.npy", np.array(fold_accuracies_lr))
np.save("Data/fold_energies_lr.npy", np.array(fold_energies_lr))
np.save("Data/accuracies_xgb.npy", np.array(accuracies_xgb))
np.save("Data/energies_xgb.npy", np.array(energies_xgb))
np.save("Data/fold_accuracies_xgb.npy", np.array(fold_accuracies_xgb))
np.save("Data/fold_energies_xgb.npy", np.array(fold_energies_xgb))

In [73]:
lr_acc_std = np.std(accuracies_lr)
lr_energy_std = np.std(energies_lr)
lr_acc_avg = np.mean(accuracies_lr)
lr_energy_avg = np.mean(energies_lr)
print(f"LR Accuracy: {lr_acc_avg:.4f} ± {lr_acc_std:.4f}")
print(f"LR Energy: {lr_energy_avg:.2f}J ± {lr_energy_std:.2f}J")

LR Accuracy: 0.5981 ± 0.0052
LR Energy: 107.34J ± 1.83J


In [74]:
xgb_acc_std = np.std(accuracies_xgb)
xgb_energy_std = np.std(energies_xgb)
xgb_acc_avg = np.mean(accuracies_xgb)
xgb_energy_avg = np.mean(energies_xgb)
print(f"XGB Accuracy: {xgb_acc_avg:.4f} ± {xgb_acc_std:.4f}")
print(f"XGB Energy: {xgb_energy_avg:.2f}J ± {xgb_energy_std:.2f}J")

XGB Accuracy: 0.5930 ± 0.0077
XGB Energy: 96.86J ± 1.32J


In [78]:
xgb_lambda_accuracies = []
xgb_lambda_energies = []

for ee in range (0, 11, 1):
    e = ee/10

    accuracies_xgb = []
    energies_xgb = []
    fold_accuracies_xgb = []
    fold_energies_xgb = []

    for i in range(repetitions):
        accuracy, energy = fold_xgb(e=e, k=5, random_state=42+i, fold_accuracies=fold_accuracies_xgb, fold_energies=fold_energies_xgb)
        
        accuracies_xgb.append(accuracy)
        energies_xgb.append(energy)
    
    
    xgb_acc_avg = np.mean(accuracies_xgb)
    xgb_energy_avg = np.mean(energies_xgb)
    
    xgb_lambda_accuracies.append(xgb_acc_avg)
    xgb_lambda_energies.append(xgb_energy_avg)
    

np.save("Data/xgb_lambda_accuracies.npy", np.array(xgb_lambda_accuracies))
np.save("Data/xgb_lambda_energies.npy", np.array(xgb_lambda_energies))

C:\Users\Michael\AppData\Local\Temp\ipykernel_129340\3496638334.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train["energy_norm"] = (X_train["avg_energy"] - energy_mean) / energy_std
C:\Users\Michael\AppData\Local\Temp\ipykernel_129340\3496638334.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test["energy_norm"]  = (X_test["avg_energy"] - energy_mean) / energy_std


Baseline: accuracy: 0.5658 | energy: 126.53J
Interpolated energy: 151.38J
Strong_model accuracy: 0.6228 | energy: 162.43J
Test set accuracy: 0.6053, avg energy: 144.63J


C:\Users\Michael\AppData\Local\Temp\ipykernel_129340\3496638334.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train["energy_norm"] = (X_train["avg_energy"] - energy_mean) / energy_std
C:\Users\Michael\AppData\Local\Temp\ipykernel_129340\3496638334.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test["energy_norm"]  = (X_test["avg_energy"] - energy_mean) / energy_std


Baseline: accuracy: 0.6360 | energy: 120.33J
Interpolated energy: 142.89J
Strong_model accuracy: 0.7149 | energy: 160.93J
Test set accuracy: 0.6798, avg energy: 138.32J


C:\Users\Michael\AppData\Local\Temp\ipykernel_129340\3496638334.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train["energy_norm"] = (X_train["avg_energy"] - energy_mean) / energy_std
C:\Users\Michael\AppData\Local\Temp\ipykernel_129340\3496638334.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test["energy_norm"]  = (X_test["avg_energy"] - energy_mean) / energy_std


...

Baseline: accuracy: 0.6053 | energy: 119.74J
Interpolated energy: 92.62J
Strong_model accuracy: 0.6623 | energy: 163.82J
Test set accuracy: 0.5702, avg energy: 81.18J


C:\Users\Michael\AppData\Local\Temp\ipykernel_129340\3496638334.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train["energy_norm"] = (X_train["avg_energy"] - energy_mean) / energy_std
C:\Users\Michael\AppData\Local\Temp\ipykernel_129340\3496638334.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test["energy_norm"]  = (X_test["avg_energy"] - energy_mean) / energy_std


Baseline: accuracy: 0.6278 | energy: 126.07J
Interpolated energy: 91.28J
Strong_model accuracy: 0.7225 | energy: 166.50J
Test set accuracy: 0.5463, avg energy: 94.02J


C:\Users\Michael\AppData\Local\Temp\ipykernel_129340\3496638334.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train["energy_norm"] = (X_train["avg_energy"] - energy_mean) / energy_std
C:\Users\Michael\AppData\Local\Temp\ipykernel_129340\3496638334.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test["energy_norm"]  = (X_test["avg_energy"] - energy_mean) / energy_std


Baseline: accuracy: 0.5815 | energy: 126.03J
Interpolated energy: 76.60J
Strong_model accuracy: 0.6035 | energy: 167.23J
Test set accuracy: 0.5551, avg energy: 95.64J
0.5606326889279437 89.4508974614333


In [98]:
#list (not numpy list) of numpy floats to list of float
#xgb_lambda_accuracies = np.load("Data/xgb_lambda_accuracies.npy").tolist()
#xgb_lambda_energies = np.load("Data/xgb_lambda_energies.npy").tolist()
print(xgb_lambda_accuracies)
print(xgb_lambda_energies)

[0.657029876977153, 0.6391915641476273, 0.6241652021089632, 0.6115992970123024, 0.598066783831283, 0.5929701230228471, 0.5838312829525484, 0.5791739894551846, 0.5753075571177504, 0.5726713532513181, 0.5681898066783833]
[145.18171622534663, 132.26659838019918, 119.01790002050379, 108.4472305633665, 101.32793220074201, 96.85676351298572, 93.20352164421008, 90.8592275405194, 89.81943813512984, 88.17966868580353, 88.02182892013278]


# Overhead

In [41]:
with open("deepseek-docker/Energy/results/BERT/20260309_163911.json", "r") as f:
    tasks_overhead = json.load(f)
    
df = pd.read_csv("deepseek-docker/Energy/results/BERT/measurement_20260309_163911.csv")

df["Time"] = pd.to_numeric(df["Time"], errors="coerce") / 1000  # Convert ms → s
df = df.dropna(subset=["Time"])


df = df.set_index("Time").sort_index()
df["GPU0_POWER (mWatts)"] = pd.to_numeric(df["GPU0_POWER (mWatts)"], errors="coerce")
df["GPU0_POWER (mWatts)"] = df["GPU0_POWER (mWatts)"].interpolate(method="linear")


results = defaultdict(lambda: {
    "task_id": None,
    "model_name": None,
    "measurements": []
})

for task in tasks_overhead:
    key = (task["task_id"], task["model_name"])
    start = float(task["start_time"])
    end = float(task["end_time"])
    duration = end - start
    

    interval = df.loc[start:end]
    avg_power_mw = interval["GPU0_POWER (mWatts)"].mean() if not interval.empty else None
    
    interval_trapz = df.loc[start:end, "GPU0_POWER (mWatts)"]
    times = interval_trapz.index.to_numpy()
    powers_watts = interval_trapz.to_numpy() / 1000
    energy_trapz = compute_energy_trapezoid(df, start, end)
    
    
    #print(avg_power_mw)
    energy_joules = (avg_power_mw / 1000) * duration if avg_power_mw is not None else None
    results[key]["task_id"] = task["task_id"]
    results[key]["model_name"] = task["model_name"]
    results[key]["measurements"].append({
        "iteration": task["iteration"],
        "start_time": round(start, 3),
        "end_time": round(end, 3),
        "duration": round(duration, 3),
        "avg_gpu_power_watts": round(avg_power_mw / 1000, 3) if avg_power_mw is not None else None,
        "energy_joules": round(energy_trapz, 3) if energy_trapz is not None else None
    })


grouped_results = list(results.values())

energy_stats = defaultdict(list)

for group in grouped_results:
    model = group["model_name"]
    for m in group["measurements"]:
        if m["energy_joules"] is not None:
            energy_stats[model].append(m["energy_joules"])

for model, energies in energy_stats.items():
    avg_energy = (sum(energies)/1138) / len(energies) if energies else 0
    max_energy = (max(energies)) if energies else 0
    sd_energy = (np.std(energies)/1138) if energies else 0
    print(f"Model: {model} | Avg Energy: {avg_energy:.5f} J")
    #print(f"Model: {model} | Max Energy: {max_energy:.5f} J")
    print(f"Model: {model} | Std Dev Energy: {sd_energy:.5f} J")


{'task_id': 'combined', 'model_name': 'microsoft/codebert-base', 'measurements': [{'iteration': 19, 'start_time': 1773071009.458, 'end_time': 1773071012.645, 'duration': 3.187, 'avg_gpu_power_watts': np.float64(120.898), 'energy_joules': np.float64(376.475)}, {'iteration': 14, 'start_time': 1773071027.645, 'end_time': 1773071030.832, 'duration': 3.187, 'avg_gpu_power_watts': np.float64(125.206), 'energy_joules': np.float64(398.848)}, {'iteration': 10, 'start_time': 1773071045.832, 'end_time': 1773071049.026, 'duration': 3.194, 'avg_gpu_power_watts': np.float64(120.76), 'energy_joules': np.float64(393.788)}, {'iteration': 26, 'start_time': 1773071064.026, 'end_time': 1773071067.22, 'duration': 3.193, 'avg_gpu_power_watts': np.float64(126.305), 'energy_joules': np.float64(408.524)}, {'iteration': 22, 'start_time': 1773071082.22, 'end_time': 1773071085.396, 'duration': 3.177, 'avg_gpu_power_watts': np.float64(129.88), 'energy_joules': np.float64(412.806)}, {'iteration': 6, 'start_time': 1

In [93]:
with open("deepseek-docker/Energy/results/Router/20260309_172447.json", "r") as f:
    tasks_overhead = json.load(f)
    
df = pd.read_csv("deepseek-docker/Energy/results/Router/measurement_20260309_172447.csv")

df["Time"] = pd.to_numeric(df["Time"], errors="coerce") / 1000  # Convert ms → s
df = df.dropna(subset=["Time"])


df = df.set_index("Time").sort_index()
results = defaultdict(lambda: {
    "task_id": None,
    "model_name": None,
    "measurements": []
})

for task in tasks_overhead:
    key = (task["task_id"], task["model_name"])
    start = float(task["start_time"])
    end = float(task["end_time"])
    duration = end - start
    end = end + 0.2 
    #interval = df[(df["Time"] >= start) & (df["Time"] <= end)]
    interval = df.loc[start:end]
    avg_power_mw = interval["GPU0_POWER (mWatts)"].mean() if not interval.empty else None
    cpu_energy = 0
    for i in range(12):
        cpu_energy += interval[f"CORE{i}_ENERGY (J)"].iloc[-1] - interval[f"CORE{i}_ENERGY (J)"].iloc[0]

    #print(avg_power_mw)
    energy_joules = (avg_power_mw / 1000) * duration if avg_power_mw is not None else None
    results[key]["task_id"] = task["task_id"]
    results[key]["model_name"] = task["model_name"]
    results[key]["measurements"].append({
        "iteration": task["iteration"],
        "start_time": round(start, 3),
        "end_time": round(end, 3),
        "duration": round(duration, 3),
        "avg_gpu_power_watts": round(avg_power_mw / 1000, 3) if avg_power_mw is not None else None,
        "energy_joules": energy_joules if energy_joules is not None else None,
        "cpu_energy_joules": cpu_energy if cpu_energy is not None else None,
        "total_energy_joules": energy_joules + cpu_energy if energy_joules is not None and cpu_energy is not None else None
    })


grouped_results = list(results.values())


energy_stats = defaultdict(list)

for group in grouped_results:
    model = group["model_name"]
    for m in group["measurements"]:
        if m["total_energy_joules"] is not None:
            energy_stats[model].append(m["total_energy_joules"])

for model, energies in energy_stats.items():
    avg_energy = (sum(energies)/1138) / len(energies) if energies else 0
    sd_energy = (np.std(energies)/1138) if energies else 0
    max_energy = (max(energies)) if energies else 0
    print(f"Model: {model} | Avg Energy: {avg_energy:.5f} J")
    print(f"Model: {model} | Energy Std Dev: {sd_energy:.5f} J")


{'task_id': 'combined', 'model_name': 'xgb', 'measurements': [{'iteration': 8, 'start_time': 1773073610.409, 'end_time': 1773073610.749, 'duration': 0.14, 'avg_gpu_power_watts': np.float64(28.394), 'energy_joules': np.float64(3.9615153498649596), 'cpu_energy_joules': np.float64(1.9882049558800645), 'total_energy_joules': np.float64(5.949720305745024)}, {'iteration': 24, 'start_time': 1773073640.576, 'end_time': 1773073640.915, 'duration': 0.139, 'avg_gpu_power_watts': np.float64(28.313), 'energy_joules': np.float64(3.921694065570831), 'cpu_energy_joules': np.float64(0.0), 'total_energy_joules': np.float64(3.921694065570831)}, {'iteration': 25, 'start_time': 1773073700.795, 'end_time': 1773073701.134, 'duration': 0.139, 'avg_gpu_power_watts': np.float64(28.56), 'energy_joules': np.float64(3.9698777027130125), 'cpu_energy_joules': np.float64(3.2768859845236875), 'total_energy_joules': np.float64(7.2467636872367)}, {'iteration': 28, 'start_time': 1773073715.934, 'end_time': 1773073716.274